Recommended VM specs for all notebooks: 4 cores, 26GB RAM.

This notebook:
* transfers data from the workspace bucket into the virtual environment
* calcualtes TPM from RNAseq readcounts
* aligns sample ids between the RNAseq/genetic/covariate data

## Setup

In [1]:
install.packages('R.utils')
BiocManager::install('AnVILGCP')
BiocManager::install('txdbmaker')

library(AnVILGCP)
library(data.table)
library(GenomicFeatures)

# Get bcftools for fast VCF file manipulation
system('git clone --depth 1 --recurse-submodules https://github.com/samtools/htslib.git')
system('git clone --depth 1 https://github.com/samtools/bcftools.git')
system('cd bcftools && make -j')

Installing package into ‘/home/jupyter/packages’
(as ‘lib’ is unspecified)

also installing the dependencies ‘R.oo’, ‘R.methodsS3’


'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cloud.r-project.org

Bioconductor version 3.21 (BiocManager 1.30.25), R 4.5.0 (2025-04-11)

Installing package(s) 'AnVILGCP'

Old packages: 'alabaster.base', 'AnnotationHub', 'AnVIL', 'base64enc', 'BH',
  'bigrquery', 'BiocFileCache', 'BiocGenerics', 'BiocManager', 'BiocParallel',
  'blob', 'boot', 'broom', 'bslib', 'cli', 'clock', 'cluster', 'colorspace',
  'commonmark', 'cowplot', 'cpp11', 'credentials', 'crosstalk', 'curl',
  'data.table', 'DBI', 'dbplyr', 'DelayedArray', 'DESeq2', 'devtools',
  'diffobj', 'digest', 'downlit', 'dplyr', 'DT', 'dtplyr', 'edgeR', 'evaluate',
  'ExperimentHub', 'fansi', 'fitdistrplus', 'forcats', 'foreign', 'fs',
  'futile.logger', 'future', 'future.a

## Get data from Terra data tables

Here we load the `anvil_file` data table (where the 1000G and MAGE data exported from [AnVIL Data explorer](https://explore.anvilproject.org/datasets) are referenced) and the `anno_metadata` table (where other annotation from various sources are referenced).

We need the expression counts, covariates, sample metadata from MAGE. The rest of this notebook is primarily concerned with aligning the ids between these files.

In addition, we grab the GENCODE annotation file so that we can estimate gene transcript lengths from their exons, to calculate TPM.

The `AnVILGCP::avtable()` function is used to query the Terra data tables for files, and downloads them. For files hosted in Google servers (such as the annotation data stored in this workspace's bucket), the files are copied over using `gcloud storage cp`. For files hosted by AnVIL such as the MAGE data imported via [AnVIL data explorer](https://explore.anvilproject.org/datasets?filter=%5B%7B%22categoryKey%22%3A%22datasets.title%22%2C%22value%22%3A%5B%22AnVIL_MAGE%22%5D%7D%5D), `tnu drs copy` is used to copy the files over from their DRS URIs.

In [2]:
get_drs_file <- function(drs) system(paste0('tnu drs copy      "',drs,'" .'))
get_gs_file  <- function(gs)  system(paste0('gcloud storage cp "',gs, '" .'))

# Get MAGE files from Terra data table created by import from the AnVIL data explorer
anvil_file_tbl <- setDT(avtable('anvil_file'))
anvil_file_tbl[
  ][`pfb:file_name` %in% c('expression.counts.csv', 'eQTL_covariates.tab.gz', 'sample.metadata.MAGE.v1.0.txt')
  ][, `pfb:file_ref` # This is the column containing DRS URIs
] |> sapply(get_drs_file)

anno_meta_tbl <- setDT(avtable('anno_metadata'))
anno_meta_tbl[
  ][file %in% grep('gencode.v49.annotation.gtf.gz', file, val=T)
  ][, file # This is the column containing gs:// URIs
] |> sapply(get_gs_file)

counts      <- fread('expression.counts.csv')
covars      <- fread('eQTL_covariates.tab.gz')
sample_meta <- fread('sample.metadata.MAGE.v1.0.txt')

Registered S3 methods overwritten by 'AnVIL':
  method                         from    
  print.avworkflow_configuration AnVILGCP
  print.gcloud_sdk_result        AnVILGCP



drs://drs.anv0:v2_d3dfbecf-cf6a-3d10-b167-dd232916b9b6 
                                                     0 
drs://drs.anv0:v2_505b300b-a1c7-3383-8ee7-becae9b11616 
                                                     0 
drs://drs.anv0:v2_9fef2e4a-000a-3f0a-936d-60009719cd8d 
                                                     0

gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/gencode.v49.annotation.gtf.gz 
                                                                                         0

"0" indicates a file was imported successfully.

## Calculate Transcripts Per Million (TPM) from read counts

In [3]:
# Get calculate gene transcript lengths by summing the lengths of GENCODE-defined exons for that gene.
txdb <- makeTxDbFromGFF('gencode.v49.annotation.gtf.gz', format="gtf")
exon_list_per_gene <- exonsBy(txdb, by='gene')
gene_length <- sum(width(reduce(exon_list_per_gene)))
gene_length <- as.data.table(gene_length, keep.rownames='gene')
counts2 <- merge(gene_length, copy(counts), by='gene')

# Calculate TPM
sample_colnames <- setdiff(names(counts2), c('gene','gene_length'))
rpkm <- counts2[, (sample_colnames) := lapply(.SD, function(x) x/(gene_length/1000)), .SDcols=sample_colnames]
scaling_factors <- colSums(rpkm[, .SD, .SDcols=sample_colnames])
tpm <- rpkm[, .SDcols=sample_colnames
            , (sample_colnames) := Map(.SD, scaling_factors, f=function(x,sf) x/(sf/1e6)) ]

tpm[, gene_length := NULL]

Warning message in call_fun_in_txdbmaker("makeTxDbFromGFF", ...):
“makeTxDbFromGFF() has moved from GenomicFeatures to the txdbmaker package,
  and is formally deprecated in GenomicFeatures >= 1.59.1. Please call
  txdbmaker::makeTxDbFromGFF() to get rid of this warning.”
Import genomic features from the file as a GRanges object ... 
OK

Prepare the 'metadata' data frame ... 
OK

Make the TxDb object ... 
Warning message in .get_cds_IDX(mcols0$type, mcols0$phase):
“The "phase" metadata column contains non-NA values for features of type
  stop_codon. This information was ignored.”
Warning message in .makeTxDb_normarg_chrominfo(chrominfo):
“genome version information is not available for this TxDb object”
OK



## Align sample IDs across datasets
The expression and genotype datasets have different id schemes. Here we convert all ids to the same format, and ensure they are in the same order.

In [4]:
# Make counts row and column order match tpm.
counts <- counts[match(tpm$gene,gene)]
identical(counts$gene,   tpm$gene)
identical(names(counts), names(tpm))

# Convert MAGE "TOR###" IDs to 1kG "NA###" and "HG###" IDs in expression data
ids_mage <- names(tpm) |> setdiff('gene')
ids_1kg  <- sample_meta[match(ids_mage, sample_coriellID), sample_kgpID]

tpm    |> setnames(old=ids_mage, new=ids_1kg)
counts |> setnames(old=ids_mage, new=ids_1kg)

all(ids_1kg %in% names(covars)) # The covariate data already has 1kG IDs

[1] TRUE

[1] TRUE

[1] TRUE

## Transpose so that samples are rows

In [5]:
counts <- transpose(counts, keep.names='sample_id', make.names='gene')
tpm    <- transpose(tpm,    keep.names='sample_id', make.names='gene')
covars <- transpose(covars, keep.names='sample_id', make.names= 'id' )

In [6]:
# Inspect the final state of the files:
counts[1:3,1:6]
tpm[1:3,1:6]
covars[1:3,1:6]

sample_id,ENSG00000000005.6,ENSG00000002745.13,ENSG00000003137.9,ENSG00000004846.17,ENSG00000004848.8
<chr>,<int>,<int>,<int>,<int>,<int>
NA06985,0,16,1,16,1
NA07000,0,9,1,17,0
NA11919,0,5,5,4,3


sample_id,ENSG00000000005.6,ENSG00000002745.13,ENSG00000003137.9,ENSG00000004846.17,ENSG00000004848.8
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
NA06985,0,0.9270782,0.03908327,0.3440181,0.06073345
NA07000,0,1.1991542,0.08987254,0.8405167,0.00000000
NA11919,0,1.0247780,0.69123279,0.3042179,0.64448468


sample_id,sex,PC1,PC2,PC3,PC4
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
HG00096,XY,-0.0216243,0.0487524,0.00362148,0.0446186
HG00100,XX,-0.0215191,0.0503942,0.00281614,0.040869
HG00105,XY,-0.0215323,0.0486066,0.0041307,0.0403912


## Write and copy to bucket

In [7]:
fwrite(counts , 'mage_expression-counts.csv'            )
fwrite(   tpm , 'mage_expression-tpm.csv'               )
fwrite(covars , 'mage_eQTL_covariates.tab.gz'           )

system('gcloud storage cp mage_expression-counts.csv    $WORKSPACE_BUCKET/data/derived/')
system('gcloud storage cp mage_expression-tpm.csv       $WORKSPACE_BUCKET/data/derived/')
system('gcloud storage cp mage_eQTL_covariates.tab.gz   $WORKSPACE_BUCKET/data/derived/')
system('gcloud storage cp gencode.v49.annotation.gtf.gz $WORKSPACE_BUCKET/data/annotation/')